https://mermaid.ai/d/58908990-2096-4e72-997d-6a769d04407b 



In [1]:
# install dependencies from requirements file (fix typo and use %pip)
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np

from config import *
from pdf_loader import read_pdf_text
from chunking import split_sections, chunk_text, build_chunks
from embeddings import load_embedder, embed_chunks
from db import init_db, upsert_chunks, delete_all_chunks, count_chunks, preview_chunks, delete_doc_chunks
from retrieval import retrieve_topk, hybrid_retrieve_topk
from rerank import rerank_results

print('✅ All modules imported successfully')


✅ All modules imported successfully


In [3]:
from db import delete_all_chunks,count_chunks,preview_chunks
count_chunks(PG_CONN_STR)

📦 Chunks (total): 2474


2474

In [4]:
delete_all_chunks(PG_CONN_STR)

🗑️  All chunks deleted from rag_cv_chunks


In [5]:
count_chunks(PG_CONN_STR)

📦 Chunks (total): 0


0

In [6]:
print("🔎 Loading embedding model...")
embedder = load_embedder(EMBED_MODEL_NAME)

dim = embedder.get_sentence_embedding_dimension()

from db import init_db

# =========================================
# 2️ INIT DATABASE
# =========================================

print("🧱 Initializing database...")
init_db(PG_CONN_STR, dim)


🔎 Loading embedding model...
🔎 Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
🧱 Initializing database...
✅ DB initialised — embedding dim=384, schema version=2


init_db (done)>> get data (done)>> split to chunks  
split to chunks 
1) split text or pdf to (sections or (chatpers and subscetion) ) يعتمد على فهمك للداتا الي معك 
2) split sections to chunks 
3) chunks content to embedder 
4) embeddeing to database
userQ >> matching with DB >> retrive top K >>> send to llm >> gen.final answer 

In [7]:
# =========================================
# INGEST PDF
# =========================================

pdf_path = 'Deep+Learning+Ian+Goodfellow.pdf'  # update path if needed
doc_name = DOC_NAME  # from config.py = 'DeepLearning-IanGoodfellow_RAG'

print('📄 Reading PDF...')
text = read_pdf_text(pdf_path)
print(f'✅ PDF loaded — {len(text):,} characters, {text.count(chr(12))} pages')


📄 Reading PDF...
✅ PDF loaded — 1,780,220 characters, 800 pages


In [8]:
# Optional: save extracted text to disk for inspection
txt_path = 'Deep_Learning_Book.txt'

with open(txt_path, 'w', encoding='utf-8') as f:
    f.write(text)

print(f'💾 Text saved to {txt_path}')


💾 Text saved to Deep_Learning_Book.txt


In [9]:
import fitz

print(fitz.__doc__)

PyMuPDF 1.27.2.2: Python bindings for the MuPDF 1.27.2 library.
Python 3.13 running on win32 (64-bit).



In [10]:
import fitz  # PyMuPDF binding: PDF/ebook/document handling (open, read, extract text)
import re

def read_pdf_text2(pdf_path: str):
    """
    Read a PDF and return cleaned text.
    fitz is used to open and parse pages for text extraction.
    """

    # Open PDF document
    doc = fitz.open(pdf_path)

    pages_text = []

    # Iterate pages and extract text
    for i, page in enumerate(doc):
        print(f"Reading page {i+1}/{len(doc)}")

        txt = page.get_text()  # extract page text

        # clean null / control chars and normalize whitespace
        txt = txt.replace("\x00", " ")
        txt = re.sub(r"[ \t]+", " ", txt)

        # keep page boundary markers
        pages_text.append(f"\n[PAGE {i+1}]\n{txt}")

    # join all pages
    text = "\n\n".join(pages_text)

    # collapse many empty lines to at most 2
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [11]:

pdf_path = r"D:\GENERATIVE & AGENTIC AI Professional\DEPI_GENERATIVE & AGENTIC AI Professional_Laps&projects\M7 RAG_Retrieval Augmented Generation\RAG - Book project\Deep+Learning+Ian+Goodfellow.pdf"
doc_name = "DeepLearning-IanGoodfellow_RAG"


In [12]:
# =========================================
# 3️ INGEST CV
# =========================================

print("📄 Reading PDF...")
text = read_pdf_text2(pdf_path)

📄 Reading PDF...
Reading page 1/801
Reading page 2/801
Reading page 3/801
Reading page 4/801
Reading page 5/801
Reading page 6/801
Reading page 7/801
Reading page 8/801
Reading page 9/801
Reading page 10/801
Reading page 11/801
Reading page 12/801
Reading page 13/801
Reading page 14/801
Reading page 15/801


Reading page 16/801
Reading page 17/801
Reading page 18/801
Reading page 19/801
Reading page 20/801
Reading page 21/801
Reading page 22/801
Reading page 23/801
Reading page 24/801
Reading page 25/801
Reading page 26/801
Reading page 27/801
Reading page 28/801
Reading page 29/801
Reading page 30/801
Reading page 31/801
Reading page 32/801
Reading page 33/801
Reading page 34/801
Reading page 35/801
Reading page 36/801
Reading page 37/801
Reading page 38/801
Reading page 39/801
Reading page 40/801
Reading page 41/801
Reading page 42/801
Reading page 43/801
Reading page 44/801
Reading page 45/801
Reading page 46/801
Reading page 47/801
Reading page 48/801
Reading page 49/801
Reading page 50/801
Reading page 51/801
Reading page 52/801
Reading page 53/801
Reading page 54/801
Reading page 55/801
Reading page 56/801
Reading page 57/801
Reading page 58/801
Reading page 59/801
Reading page 60/801
Reading page 61/801
Reading page 62/801
Reading page 63/801
Reading page 64/801
Reading page 65/801


In [13]:
# =========================================
# SAVE TEXT FILE
# =========================================

txt_path = r"D:\GENERATIVE & AGENTIC AI Professional\DEPI_GENERATIVE & AGENTIC AI Professional_Laps&projects\M7 RAG_Retrieval Augmented Generation\RAG - Book project\Deep_Learning_Book_2.txt"

with open(txt_path, "w", encoding="utf-8") as f:
    f.write(text)

print("✅ Text saved to:", txt_path)

✅ Text saved to: D:\GENERATIVE & AGENTIC AI Professional\DEPI_GENERATIVE & AGENTIC AI Professional_Laps&projects\M7 RAG_Retrieval Augmented Generation\RAG - Book project\Deep_Learning_Book_2.txt


In [14]:
# READ TEXT FILE
# =========================================

txt_path = r"D:\GENERATIVE & AGENTIC AI Professional\DEPI_GENERATIVE & AGENTIC AI Professional_Laps&projects\M7 RAG_Retrieval Augmented Generation\RAG - Book project\Deep_Learning_Book_2.txt"
Text = ""

with open(txt_path, "r", encoding="utf-8") as f:
    Text = f.read()

print("✅ Text loaded from:", txt_path)

✅ Text loaded from: D:\GENERATIVE & AGENTIC AI Professional\DEPI_GENERATIVE & AGENTIC AI Professional_Laps&projects\M7 RAG_Retrieval Augmented Generation\RAG - Book project\Deep_Learning_Book_2.txt


In [15]:
text[:500]

'[PAGE 1]\n\n[PAGE 2]\nDeep Learning\nIan Goodfellow\nYoshua Bengio\nAaron Courville\n\n[PAGE 3]\nContents\nWebsite\nvii\nAcknowledgments\nviii\nNotation\nxi\n1\nIntroduction\n1\n1.1\nWho Should Read This Book? . . . . . . . . . . . . . . . . . . . .\n8\n1.2\nHistorical Trends in Deep Learning . . . . . . . . . . . . . . . . .\n11\nI\nApplied Math and Machine Learning Basics\n29\n2\nLinear Algebra\n31\n2.1\nScalars, Vectors, Matrices and Tensors . . . . . . . . . . . . . . .\n31\n2.2\nMultiplying Matrices and Vectors . . . . . . .'

In [16]:
print("✂️ Splitting sections...")
sections = split_sections(text)
len(sections)

✂️ Splitting sections...


950

In [17]:
sections

[{'chapter': 'front_matter',
  'section': 'general',
  'content': '[PAGE 1]\n[PAGE 2]\nDeep Learning\nIan Goodfellow\nYoshua Bengio\nAaron Courville\n[PAGE 3]\nContents\nWebsite\nvii\nAcknowledgments\nviii\nNotation\nxi\n1\nIntroduction\n1\n1.1\nWho Should Read This Book? . . . . . . . . . . . . . . . . . . . .\n8\n1.2\nHistorical Trends in Deep Learning . . . . . . . . . . . . . . . . .\n11\nI\nApplied Math and Machine Learning Basics\n29\n2\nLinear Algebra\n31\n2.1\nScalars, Vectors, Matrices and Tensors . . . . . . . . . . . . . . .\n31\n2.2\nMultiplying Matrices and Vectors . . . . . . . . . . . . . . . . . .\n34\n2.3\nIdentity and Inverse Matrices\n. . . . . . . . . . . . . . . . . . . .\n36\n2.4\nLinear Dependence and Span\n. . . . . . . . . . . . . . . . . . . .\n37\n2.5\nNorms . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .\n39\n2.6\nSpecial Kinds of Matrices and Vectors\n. . . . . . . . . . . . . . .\n40\n2.7\nEigendecomposition . . . . . . . . . . . . . . . 

-- chapters (col) 
-- subsection in chapter (col)

In [18]:
# =========================================
# CHUNK SECTIONS
# =========================================
# Chunk size is capped at 220 tokens (256 is the hard limit for all-MiniLM-L6-v2)
# Setting it higher causes SILENT truncation during embedding — do not exceed 220.

print('✂️ Building chunks...')
chunks = build_chunks(
    raw_text=text,
    embed_model_name=EMBED_MODEL_NAME,
    chunk_size=CHUNK_SIZE,      # 220 from config
    overlap=CHUNK_OVERLAP,      # 40  from config
)
print(f'✅ {len(chunks)} chunks created')
print(f'   Sample: {chunks[0]}')


✂️ Building chunks...


Token indices sequence length is longer than the specified maximum sequence length for this model (7776 > 512). Running this sequence through the model will result in indexing errors


✅ 2474 chunks created
   Sample: {'chapter': 'front_matter', 'section': 'general', 'entry_id': 0, 'chunk_in_entry': 0, 'token_len': 220, 'content': '[ page 1 ] [ page 2 ] deep learning ian goodfellow yoshua bengio aaron courville [ page 3 ] website vii acknowledgments viii notation xi introduction 1. 1 who should read this book?.................... 1. 2 historical trends in deep learning................. i applied math and machine learning basics linear algebra 2. 1 scalars, vectors, matrices and tensors............... 2. 2 multiplying matrices and vectors.................. 2. 3 identity and inverse matrices.................... 2. 4 linear dependence and span.................... 2. 5 norms......', 'page_start': 1, 'page_end': 1}


In [19]:
# =========================================
# CREATE EMBEDDINGS
# =========================================

print('🧠 Creating embeddings...')
# embed_chunks accepts both List[dict] and List[str]
vectors = embed_chunks(chunks, embedder)
print(f'✅ Embeddings created — shape: {vectors.shape}')
print(f'   Sample vector (first 5 dims): {vectors[0][:5]}')


🧠 Creating embeddings...


Batches:   0%|          | 0/78 [00:00<?, ?it/s]

✅ Embeddings created — shape: (2474, 384)
   Sample vector (first 5 dims): [-0.05744905 -0.0914406  -0.04047587 -0.03177961 -0.03230645]


In [20]:
vectors[:2]

array([[-5.74490465e-02, -9.14406031e-02, -4.04758677e-02,
        -3.17796133e-02, -3.23064514e-02,  3.73383425e-02,
        -7.03019788e-03, -8.69304128e-03, -8.72745961e-02,
        -1.87583808e-02,  8.66848882e-03,  5.88914268e-02,
         1.30030466e-03, -3.72456871e-02, -1.13166697e-01,
         9.58171189e-02, -9.75994207e-03,  2.12462377e-02,
        -1.04789741e-01, -7.36549869e-03,  1.99990720e-02,
         7.13930745e-03, -4.09739874e-02, -4.26042750e-02,
         5.00148199e-02,  3.79393324e-02,  8.13179314e-02,
        -4.25469801e-02, -2.90862713e-02,  2.71987487e-02,
         3.88678536e-03,  1.45752467e-02,  5.50990552e-02,
        -5.92438467e-02, -1.11231744e-01,  3.88359907e-03,
         5.05873859e-02,  2.33657919e-02, -2.12947093e-02,
         2.61428244e-02,  2.24870984e-02, -1.74215510e-02,
         5.00744060e-02,  4.88202088e-02,  7.32574314e-02,
         3.34492736e-02, -1.86621286e-02, -1.14557952e-01,
         3.98319587e-02, -2.00854372e-02, -6.62531108e-0

In [21]:
import psycopg
with psycopg.connect(PG_CONN_STR) as c:
    with c.cursor() as cur:
        cur.execute("""
            SELECT column_name FROM information_schema.columns
            WHERE table_name='rag_cv_chunks'
            ORDER BY ordinal_position;
        """)
        print(cur.fetchall())

[('id',), ('doc_name',), ('section',), ('chunk_index',), ('content',), ('embedding',), ('chapter',), ('content_tsv',)]


In [22]:
from db import delete_all_chunks, init_db
delete_all_chunks(PG_CONN_STR)
init_db(PG_CONN_STR, dim)

🗑️  All chunks deleted from rag_cv_chunks
✅ DB initialised — embedding dim=384, schema version=2


In [23]:
# ========================================
# 6️ STORE IN DATABASE
# =========================================

import importlib
import sql_queries
import db

# Force reload both modules to pick up the new SQL query
importlib.reload(sql_queries)
importlib.reload(db)

from db import upsert_chunks, count_chunks, preview_chunks

# Now run upsert
upsert_chunks(PG_CONN_STR, doc_name, chunks, vectors)
count_chunks(PG_CONN_STR)

💾 Stored 2474 chunks for 'DeepLearning-IanGoodfellow_RAG'
📦 Chunks (total): 2474


2474

In [24]:
import importlib
import sql_queries
import db

# Force reload with updated preview_chunks (no chapter column)
importlib.reload(sql_queries)
importlib.reload(db)

from db import count_chunks, preview_chunks

In [25]:
count_chunks(PG_CONN_STR)
preview_chunks(PG_CONN_STR)

📦 Chunks (total): 2474

🔎 Stored Chunks Preview

DOC:     DeepLearning-IanGoodfellow_RAG
CHAPTER: front_matter
SECTION: general
CHUNK:   0
[ page 1 ] [ page 2 ] deep learning ian goodfellow yoshua bengio aaron courville [ page 3 ] website vii acknowledgments viii notation xi introduction 1. 1 who should read this book?...................
----------------------------------------
DOC:     DeepLearning-IanGoodfellow_RAG
CHAPTER: front_matter
SECTION: general
CHUNK:   1
... 2. 4 linear dependence and span.................... 2. 5 norms................................. 2. 6 special kinds of matrices and vectors............... 2. 7 eigendecomposition...................
----------------------------------------
DOC:     DeepLearning-IanGoodfellow_RAG
CHAPTER: front_matter
SECTION: general
CHUNK:   2
2. 10 the trace operator......................... 2. 11 the determinant........................... 2. 12 example : principal components analysis............. probability and information theory 3. 1

In [26]:
import os
from openai import OpenAI
from config import HF_API_KEY, HF_BASE_URL, HF_MODEL_NAME

# Set your real token: either edit config.py or set env var HF_API_KEY
client = OpenAI(
    base_url=HF_BASE_URL,
    api_key=HF_API_KEY,   # read from config / env — never a literal space
)

def generate_text(context, question,
                  model=HF_MODEL_NAME,
                  temperature=0.2,
                  max_tokens=1024):

    prompt = f'''
You are an AI assistant that answers questions strictly using the provided CONTEXT.

STRICT RULES:
- Use ONLY the information inside the CONTEXT.
- Do NOT use prior knowledge or external information.
- If the answer is not in the CONTEXT, say: "The answer is not found in the provided context."

Answer style: concise, bullet points for lists.

Output format:
Answer:
<your answer>

Source:
<section name>

CONTEXT:
{context}

QUESTION:
{question}
'''

    try:
        completion = client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        return f'Error generating response: {str(e)}'


In [28]:
reranked, answer = ask_rag_question(
    "What are the main parts of the Deep Learning book?",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text
)

In [29]:
reranked, answer = ask_rag_question(
    "What chapter discusses autoencoders?",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text
)

In [30]:
reranked, answer = ask_rag_question(
    "What is backpropagation?",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text
)

In [31]:
reranked[0]

('6.5.9_diﬀerentiation_outside_the_deep_learning_community',
 'the deep learning community has been somewhat isolated from the broader computer science community and has largely developed its own cultural attitudes [ page 238 ] chapter 6. deep feedforward networks concerning how to perform. more generally, the ﬁeld of automatic is concerned with how to compute derivatives algorithmically. the back - propagation algorithm described here is only one approach to automatic. it is a special case of a broader class of techniques called reverse mode accumulation. other approaches evaluate the subexpressions of the chain rule in orders. in general, determining the order of evaluation that results in the lowest computational cost is a problem. finding the optimal sequence of operations to compute the gradient is np - complete (, ), naumann 2008 in the sense that it may require simplifying algebraic expressions into their least expensive form. for example, suppose we have variables p1, p2,..., p

In [32]:
reranked, answer = ask_rag_question(
    "Explain gradient descent.",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text
)

In [33]:
reranked, answer = ask_rag_question(
    "Explain stochastic gradient descent.",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text
)

In [34]:
# search in db for "stochastic gradient descent" without embedding (hybrid search)
reranked, answer = ask_rag_question(
    "Explain stochastic gradient descent.",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text,
    use_hybrid=True
)

In [35]:

# search in db for "What is backpropagation?" without embedding (hybrid search)
reranked, answer = ask_rag_question(
    "What is backpropagation?",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text,
    use_hybrid=True
)

In [36]:
# Try a question that requires synthesis across multiple sections
reranked, answer = ask_rag_question(
    "How do convolutional neural networks differ from recurrent neural networks?",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text,
    use_hybrid=True
)

In [37]:
# try a question that likely requires broader context
reranked, answer = ask_rag_question(
    "What are the main challenges in training deep neural networks?",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text,
    use_hybrid=True
)

In [38]:
# search for quesion with using llm
reranked, answer = ask_rag_question(
    "What are the main challenges in training deep neural networks?",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text,
    use_hybrid=False  # standard vector search only
)

In [39]:
# search for a specific term that may not be in the text (to test "not found" response)
reranked, answer = ask_rag_question(
    "What is the meaning of life according to the book?",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text,
    use_hybrid=True
)

In [ ]:
# try a quesion with using metadata


TypeError: ask_rag_question() got an unexpected keyword argument 'condition'